# What is Persistence
Persistence in LangGraph refers to the ability to save and restore the state of a workflow over time

## Use case of persistence 
- Implementing Short Term Memory
    - Resuming Conversation 
- Fault Tolerance
    - if workflow crashes at a particular node we need not run the entire workflow , we can simply run it from the exact node where it crashed
- Human In The Loops
- Time Travel 

In [120]:
from langchain_core.messages import HumanMessage, AIMessage,SystemMessage 
from langchain_core.prompts import PromptTemplate 
from langchain_groq import ChatGroq 
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
load_dotenv()

True

In [121]:
llm= ChatGroq(
    model= "qwen/qwen3.6-27b",
    streaming=True,
    temperature=1,
    reasoning_format='parsed'
)

In [122]:
from typing import TypedDict
class JokeState(TypedDict):
    topic:str
    joke : str
    explaination: str

In [123]:
from langgraph.graph import StateGraph , END, START

In [124]:
def joke_gen(state: JokeState)->JokeState:
    topic = state['topic']
    prompt = f"generate a joke on the topic : {topic}"
    result = llm.invoke(prompt).content
    return {
        'joke':result
    }

def joke_exp(state: JokeState)->JokeState:
    topic = state['joke']
    prompt = f"generate explaination for this joke : {topic}"
    result = llm.invoke(prompt).content
    return {
        'explaination':result
    }

In [125]:
graph = StateGraph(JokeState)

graph.add_node('joke_gen',joke_gen)
graph.add_node('joke_exp',joke_exp)

graph.add_edge(START,'joke_gen')
graph.add_edge('joke_gen','joke_exp')
graph.add_edge('joke_exp',END)

In [126]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [127]:
config1 = {'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'donal trump'},config=config1)

{'topic': 'donal trump',
 'joke': 'Donald Trump gets into an elevator and presses the button for the top floor. A few seconds later, the elevator starts going down. He keeps mashing the button while shouting, "Wrong way! Wrong way! Total disaster!"\n\nA bystander tries to be helpful and says, "Sir, I think you just pressed the down button."\n\nTrump looks at the bystander, puts on his sunglasses, and replies, "That\'s exactly what the broken elevator industry wants you to believe! The down button is fake news. We\'re going to have the greatest elevator system anyone has ever seen, and believe me, the other elevators are going to pay for this ride!"',
 'explaination': 'This joke is a piece of **political satire** that parodies Donald Trump’s distinctive public speaking style, recurring rhetorical catchphrases, and habitual pattern of reframing setbacks as external conspiracies or manufactured failures. Rather than making a serious political claim, it uses exaggeration and incongruity to

In [128]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'donal trump', 'joke': 'Donald Trump gets into an elevator and presses the button for the top floor. A few seconds later, the elevator starts going down. He keeps mashing the button while shouting, "Wrong way! Wrong way! Total disaster!"\n\nA bystander tries to be helpful and says, "Sir, I think you just pressed the down button."\n\nTrump looks at the bystander, puts on his sunglasses, and replies, "That\'s exactly what the broken elevator industry wants you to believe! The down button is fake news. We\'re going to have the greatest elevator system anyone has ever seen, and believe me, the other elevators are going to pay for this ride!"', 'explaination': 'This joke is a piece of **political satire** that parodies Donald Trump’s distinctive public speaking style, recurring rhetorical catchphrases, and habitual pattern of reframing setbacks as external conspiracies or manufactured failures. Rather than making a serious political claim, it uses exaggeration

In [129]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'donal trump', 'joke': 'Donald Trump gets into an elevator and presses the button for the top floor. A few seconds later, the elevator starts going down. He keeps mashing the button while shouting, "Wrong way! Wrong way! Total disaster!"\n\nA bystander tries to be helpful and says, "Sir, I think you just pressed the down button."\n\nTrump looks at the bystander, puts on his sunglasses, and replies, "That\'s exactly what the broken elevator industry wants you to believe! The down button is fake news. We\'re going to have the greatest elevator system anyone has ever seen, and believe me, the other elevators are going to pay for this ride!"', 'explaination': 'This joke is a piece of **political satire** that parodies Donald Trump’s distinctive public speaking style, recurring rhetorical catchphrases, and habitual pattern of reframing setbacks as external conspiracies or manufactured failures. Rather than making a serious political claim, it uses exaggeratio

In [130]:
config2 = {'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'narendra modi'},config=config2)

{'topic': 'narendra modi',
 'joke': '**Q:** Why did PM Modi keep a yoga mat in the office?  \n**A:** Because he believes in *flexible* scheduling and *core* priorities! 🧘\u200d♂️✨\n\n*(Light, positive, and inspired by his well-known public advocacy for yoga and wellness!)*',
 'explaination': 'Here is an explanation of the joke, broken down by its context and wordplay:\n\n### 1. The Real-World Context\nThe joke is based on the public profile of **Prime Minister Narendra Modi** of India. He is globally recognized as a strong advocate for **yoga**. He launched **International Yoga Day** (celebrated on June 21st) and frequently encourages people worldwide to practice yoga for physical and mental wellness. Because of this, yoga is a signature part of his public brand.\n\n### 2. The Wordplay (Puns)\nThe humor relies on **double meanings** where terms used in yoga are applied to the responsibilities of a Prime Minister.\n\n*   **"Flexible"**:\n    *   **Yoga context:** yoga requires **physica

In [131]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'narendra modi', 'joke': '**Q:** Why did PM Modi keep a yoga mat in the office?  \n**A:** Because he believes in *flexible* scheduling and *core* priorities! 🧘\u200d♂️✨\n\n*(Light, positive, and inspired by his well-known public advocacy for yoga and wellness!)*', 'explaination': 'Here is an explanation of the joke, broken down by its context and wordplay:\n\n### 1. The Real-World Context\nThe joke is based on the public profile of **Prime Minister Narendra Modi** of India. He is globally recognized as a strong advocate for **yoga**. He launched **International Yoga Day** (celebrated on June 21st) and frequently encourages people worldwide to practice yoga for physical and mental wellness. Because of this, yoga is a signature part of his public brand.\n\n### 2. The Wordplay (Puns)\nThe humor relies on **double meanings** where terms used in yoga are applied to the responsibilities of a Prime Minister.\n\n*   **"Flexible"**:\n    *   **Yoga context:** yoga

In [133]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'narendra modi', 'joke': '**Q:** Why did PM Modi keep a yoga mat in the office?  \n**A:** Because he believes in *flexible* scheduling and *core* priorities! 🧘\u200d♂️✨\n\n*(Light, positive, and inspired by his well-known public advocacy for yoga and wellness!)*', 'explaination': 'Here is an explanation of the joke, broken down by its context and wordplay:\n\n### 1. The Real-World Context\nThe joke is based on the public profile of **Prime Minister Narendra Modi** of India. He is globally recognized as a strong advocate for **yoga**. He launched **International Yoga Day** (celebrated on June 21st) and frequently encourages people worldwide to practice yoga for physical and mental wellness. Because of this, yoga is a signature part of his public brand.\n\n### 2. The Wordplay (Puns)\nThe humor relies on **double meanings** where terms used in yoga are applied to the responsibilities of a Prime Minister.\n\n*   **"Flexible"**:\n    *   **Yoga context:** yog

In [134]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'donal trump', 'joke': 'Donald Trump gets into an elevator and presses the button for the top floor. A few seconds later, the elevator starts going down. He keeps mashing the button while shouting, "Wrong way! Wrong way! Total disaster!"\n\nA bystander tries to be helpful and says, "Sir, I think you just pressed the down button."\n\nTrump looks at the bystander, puts on his sunglasses, and replies, "That\'s exactly what the broken elevator industry wants you to believe! The down button is fake news. We\'re going to have the greatest elevator system anyone has ever seen, and believe me, the other elevators are going to pay for this ride!"', 'explaination': 'This joke is a piece of **political satire** that parodies Donald Trump’s distinctive public speaking style, recurring rhetorical catchphrases, and habitual pattern of reframing setbacks as external conspiracies or manufactured failures. Rather than making a serious political claim, it uses exaggeratio